# Meta-learning (MAML)

_Few-shot & Transfer Learning_

**Don't learn one task. Learn a starting point that adapts to a brand-new task in a few steps.**

Normal training learns to do one task well.

       Meta-learning ("learning to learn") aims higher. It learns a good starting point for the weights. From that start, a brand-new task can be learned in just a few gradient steps.

---

This notebook is a practice scaffold for the **Meta-learning (MAML)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Visualize the data & results

_On real handwritten digits, does a meta-learned starting point adapt to a brand-new task much faster than a random start?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression

digits = load_digits()                   # 1797 real 8x8 handwritten digits
X = digits.data / 16.0                    # pixels scaled to 0..1
y = digits.target

# Several binary "tasks": one digit vs the rest. They share useful structure,
# so a start learned across them transfers to a brand-new task.
digits_list = [0, 1, 2, 3, 4, 6, 7, 9]
held_out = 8                              # brand-new task: 8 vs the rest
train_digits = [d for d in digits_list if d != held_out]

def make_task(d, seed=0):
    rng = np.random.default_rng(seed)
    pos = np.where(y == d)[0]
    neg = rng.choice(np.where(y != d)[0], size=len(pos), replace=False)  # balance
    idx = np.r_[pos, neg]
    return X[idx], (y[idx] == d).astype(float)

# META-INIT: average the learned weights across the OTHER tasks (a Reptile-like
# stand-in for MAML's meta-trained initialization).
Ws = []
for d in train_digits:
    Xt, yt = make_task(d)
    lr = LogisticRegression(C=1.0, max_iter=1000).fit(Xt, yt)
    Ws.append(np.r_[lr.coef_.ravel(), lr.intercept_])
meta = np.mean(Ws, axis=0)

# Held-out task: tiny SUPPORT set (few-shot) to adapt on, big QUERY set to test on.
rng = np.random.default_rng(7)
pos = np.where(y == held_out)[0]
neg = rng.choice(np.where(y != held_out)[0], size=len(pos), replace=False)
idx = np.r_[pos, neg]; rng.shuffle(idx)
Xt, yt = X[idx], (y[idx] == held_out).astype(float)
n_support = 10                           # only 10 labeled support examples
Xs_b = np.c_[Xt[:n_support], np.ones(n_support)]; ys = yt[:n_support]
Xq_b = np.c_[Xt[n_support:], np.ones(len(Xt) - n_support)]; yq = yt[n_support:]

def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

def adapt(w0, steps=20, alpha=0.3):      # manual gradient steps = MAML inner loop
    w = w0.copy(); errs = []
    for _ in range(steps + 1):
        errs.append(float(np.mean((sigmoid(Xq_b @ w) > 0.5) != yq)))   # query error
        g = Xs_b.T @ (sigmoid(Xs_b @ w) - ys) / n_support
        w = w - alpha * g
    return errs

meta_curve = adapt(meta)                  # warm start from the meta-init
cold_curve = adapt(np.zeros_like(meta))   # cold start from zeros
print("meta:", [round(e, 3) for e in meta_curve])   # ~0.50 -> ~0.20 in a few steps
print("cold:", [round(e, 3) for e in cold_curve])   # still ~0.33 at step 20

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Why does MAML optimize the loss at the adapted weights $\theta'$ instead of at the shared start $\theta$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the goal: a start that is easy to adapt from, not a start that is already good on its own. — _A great start that cannot adapt is useless for new tasks._
- Each task first runs the inner loop: $\theta' = \theta - \alpha\,\nabla_\theta\,\mathcal{L}_{\text{task}}(\theta)$. — _This is the few-step adaptation we actually use at test time._
- Measure success at $\theta'$, then push $\theta$ to make that post-adaptation loss small. — _Scoring at $\theta'$ rewards a start that adapts well, which is the whole point._

**Answer:** Because we care about performance after a quick adaptation. Scoring at $\theta'$ makes the outer loop search for a start that is fast to adapt, not one that is already good before adapting.

</details>

**Problem 2.** An inner step uses $\alpha = 0.1$ and gradient $\nabla_\theta\,\mathcal{L}_{\text{task}}(\theta) = 4$. What is the adapted weight $\theta'$ if the start is $\theta = 2$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the inner-loop rule: $\theta' = \theta - \alpha\,\nabla_\theta\,\mathcal{L}_{\text{task}}(\theta)$. — _This is the one gradient step that adapts to the task._
- Plug in: $\theta' = 2 - 0.1\times 4$. — _Substitute the given start, step size, and gradient._
- Compute: $2 - 0.4 = 1.6$. — _Step downhill by $\alpha$ times the gradient._

**Answer:** $\theta' = 2 - 0.1\times 4 = 1.6$.

</details>

**Problem 3.** What does first-order MAML (FOMAML) drop, and how does Reptile avoid the same cost?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that $\theta'$ depends on $\theta$ through a gradient, so the outer gradient has a second-derivative term. — _Second derivatives are the expensive part of full MAML._
- FOMAML simply ignores that second-derivative term. — _It treats $\theta'$ as if it did not depend on $\theta$, which is far cheaper._
- Reptile skips gradients-of-gradients entirely: adapt to a task, then move $\theta$ toward the adapted $\theta'$. — _Moving toward $\theta'$ approximates the same effect without any second derivatives._

**Answer:** FOMAML drops the second-derivative (gradient-of-gradient) term from the outer update. Reptile avoids it by adapting on a task and then nudging the shared start toward the adapted weights, needing no second derivatives at all.

</details>